# Slide 1 - Early Warning for Student Attrition

## An Explainable Machine Learning Approach with Fairness Evaluation

### Technical Presentation

**Capstone Step 6**

Two-stage early-warning framework:
- Enrollment
- Semester 1

# Slide 2 — Data, Target & Evaluation

## Dataset

**UCI — Predict Students' Dropout and Academic Success**

- **4,424** student records
- **36 predictors + 1 target**
- **0 missing values**
- **0 duplicate records**

## Target Definition

Original classes: `Dropout`, `Enrolled`, `Graduate`

- **1 = Dropout**
- **0 = Enrolled + Graduate**

> `Enrolled` students are not confirmed eventual non-dropouts, so this binary target must be interpreted carefully.

## Evaluation

**Primary:** Dropout Recall  
**Supporting:** Precision, F1, ROC-AUC, PR-AUC  
**Operational:** Recall@Top10%

# Slide 3 — Leakage-Safe Prediction Design

## Enrollment Model

Uses **22 predictors** available at or near enrollment:

- demographics and application data
- prior academic information
- macroeconomic context

Excludes:

- Semester 1 results
- all Semester 2 variables
- `Debtor`
- `Tuition fees up to date`

The last two were excluded because their exact availability at the enrollment prediction point could not be verified.

## Semester 1 Model

Uses the same **22 Enrollment predictors**, plus:

- **6 raw Semester 1 variables**
- **5 engineered features**

**Total: 33 predictors**

Engineered features include approval rate, completion gap, and no-approved-units indicators.

> Each model only uses information available at its prediction point.

# Slide 4 - Experimental Design

## Leakage-Safe Evaluation

- **80/20 stratified train/test split**
- **3,539** training students
- **885** held-out test students
- `random_state = 42`

```python
train_idx, test_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.20,
    random_state=42,
    stratify=y
)
```

## Model Selection

Training data only:

- Stratified **5-fold cross-validation**
- Logistic Regression
- Random Forest
- Gradient Boosting
- Hyperparameter tuning
- L1 feature selection and PCA

> The test set was opened only after model selection was complete.

# Slide 5 — Model Selection

## Tuned Logistic Regression

| Stage | CV Recall | Precision | F1 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|
| Enrollment | **70.6%** | 52.4% | 60.1% | 76.9% | 61.4% |
| Semester 1 | **77.1%** | 69.2% | 72.9% | 88.3% | 82.3% |

```python
search = GridSearchCV(
    estimator=pipeline,
    param_grid=logistic_param_grid,
    scoring=scoring,
    refit="Recall",
    cv=cv,
    n_jobs=-1
)

search.fit(X_train, y_train)
```

## Why Logistic Regression?

- strongest balance of **Dropout Recall** and supporting metrics
- interpretable coefficients
- simpler and more reproducible than ensemble alternatives

> Hyperparameters were selected using training-set cross-validation—not the held-out test set.

# Slide 6 — Held-Out Test Performance

| Stage | Recall | Precision | F1 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|
| Enrollment | **72.2%** | 52.6% | 60.8% | 78.6% | 64.5% |
| Semester 1 | **77.5%** | 69.8% | 73.5% | 89.1% | 83.4% |

## Confusion Matrix

| Stage | TP | FN | FP | TN |
|---|---:|---:|---:|---:|
| Enrollment | 205 | 79 | 185 | 416 |
| Semester 1 | 220 | 64 | 95 | 506 |

## Semester 1 Improvement

Compared with Enrollment:

- **+15** true positives
- **−15** false negatives
- **−90** false positives

> Semester 1 improves both dropout detection and targeting efficiency.

# Slide 7 — Explainability: What Drives the Models?

## Enrollment Model

Strongest permutation-importance signals:

- scholarship status
- course
- application mode
- gender
- admission grade
- age at enrollment

## Semester 1 Model

Academic progress becomes more influential:

- curricular units approved
- Semester 1 grade
- curricular units enrolled
- approval rate
- no-approved-units indicator
- completion gap

## Partial Dependence

Predicted dropout risk generally decreases as:

- approved Semester 1 units increase
- Semester 1 grades improve

> The model shifts from background factors toward observed academic progress. These relationships are predictive, not causal.

# Slide 8 — Fairness Audit

Fairness was evaluated on the held-out test set across:

- Gender
- Age groups
- Scholarship status

## Dropout Recall Gap

| Group | Enrollment | Semester 1 |
|---|---:|---:|
| Gender | **29.1 pts** | **9.7 pts** |
| Age | **45.4 pts** | **27.7 pts** |
| Scholarship status | **65.0 pts** | **40.2 pts** |

## Equalized-Odds Gap

| Group | Enrollment | Semester 1 |
|---|---:|---:|
| Gender | 0.340 | **0.108** |
| Age | 0.544 | **0.277** |
| Scholarship status | 0.650 | **0.402** |

> Semester 1 reduces every observed subgroup gap, but meaningful disparities remain—especially for age and scholarship status.

**Implication:** subgroup performance must be monitored and mitigated before deployment.

# Slide 9 — Limited-Capacity Intervention

Assume support is available for only the highest-risk **10% of students**.

Held-out test set: **885 students**  
Intervention capacity: **89 students**

| Stage | Dropouts Captured | Recall@Top10 | Precision@Top10 |
|---|---:|---:|---:|
| Enrollment | **73** | 25.7% | 82.0% |
| Semester 1 | **88** | 31.0% | 98.9% |

## Interpretation

### Enrollment
- earlier warning
- captures 73 Dropout cases
- useful for broader, lower-cost support

### Semester 1
- captures 88 Dropout cases
- almost all intervention slots correspond to actual Dropout cases
- better for higher-cost, targeted support

> The model’s practical value is helping prioritize limited intervention capacity.

**Caution:** Future-cohort validation is still required.

# Slide 10 — Conclusions, Limitations & Next Steps

## Conclusions

- Dropout risk can already be identified at Enrollment
- Semester 1 substantially improves predictive performance
- Later predictions rely more on observed academic progress
- Semester 1 also reduces several observed fairness gaps
- A two-stage approach provides both earlier warning and stronger later targeting

## Limitations

- Single dataset and institutional context
- `Enrolled` students are included in the 0 class
- Predictive relationships are not causal
- Some fairness subgroups have limited positive-case counts
- Future-cohort performance is unknown

## Before Deployment

1. Validate on future student cohorts
2. Define intervention capacity and operating thresholds
3. Test fairness-mitigation strategies
4. Monitor subgroup Recall and false-negative rates
5. Maintain human review for intervention decisions

## Current Status

**Promising decision-support prototype — not deployment-ready**

> Use the model to support student outreach, not to automate decisions about students.

In [1]:
!git clone -q https://github.com/rayvaril/student-dropout-capstone.git /content/capstone_slides
print("Cloned.")

Cloned.


In [1]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/capstone_slides/presentations/"
    "06A_Technical_Presentation_Student_Attrition.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

slide_count = 0

for cell in notebook["cells"]:
    cell.setdefault("metadata", {})
    cell["metadata"].setdefault("slideshow", {})

    if cell["cell_type"] == "markdown" and slide_count < 10:
        cell["metadata"]["slideshow"]["slide_type"] = "slide"
        slide_count += 1
    else:
        cell["metadata"]["slideshow"]["slide_type"] = "skip"

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=1, ensure_ascii=False)

print(f"Marked {slide_count} cells as Reveal.js slides.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/capstone_slides/presentations/06A_Technical_Presentation_Student_Attrition.ipynb'

In [2]:
!find /content/capstone_slides/presentations -maxdepth 1 -type f -print

find: ‘/content/capstone_slides/presentations’: No such file or directory


In [3]:
!ls -la /content/capstone_slides

ls: cannot access '/content/capstone_slides': No such file or directory


In [4]:
!git clone https://github.com/rayvaril/student-dropout-capstone.git /content/capstone_slides

Cloning into '/content/capstone_slides'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 200 (delta 60), reused 34 (delta 34), pack-reused 105 (from 1)
Receiving objects: 100% (200/200), 953.67 KiB | 7.17 MiB/s, done.
Resolving deltas: 100% (90/90), done.


In [5]:
!ls -la /content/capstone_slides/presentations

total 48
drwxr-xr-x  2 root root  4096 Oct  6 13:08 .
drwxr-xr-x 10 root root  4096 Oct  6 13:08 ..
-rw-r--r--  1 root root 13291 Oct  6 13:08 06A_Technical_Presentation_Student_Attrition.ipynb
-rw-r--r--  1 root root  8946 Oct  6 13:08 business_presentation_outline.md
-rw-r--r--  1 root root     1 Oct  6 13:08 .gitkeep
-rw-r--r--  1 root root  5503 Oct  6 13:08 technical_presentation_outline.md


In [6]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/capstone_slides/presentations/"
    "06A_Technical_Presentation_Student_Attrition.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

slide_count = 0

for cell in notebook["cells"]:
    cell.setdefault("metadata", {})
    cell["metadata"].setdefault("slideshow", {})

    if cell["cell_type"] == "markdown" and slide_count < 10:
        cell["metadata"]["slideshow"]["slide_type"] = "slide"
        slide_count += 1
    else:
        cell["metadata"]["slideshow"]["slide_type"] = "skip"

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=1, ensure_ascii=False)

print(f"Marked {slide_count} cells as Reveal.js slides.")

Marked 10 cells as Reveal.js slides.


In [7]:
!jupyter nbconvert \
  --to slides \
  /content/capstone_slides/presentations/06A_Technical_Presentation_Student_Attrition.ipynb \
  --output-dir /content/capstone_slides/presentations

[NbConvertApp] Converting notebook /content/capstone_slides/presentations/06A_Technical_Presentation_Student_Attrition.ipynb to slides
[NbConvertApp] Writing 286981 bytes to /content/capstone_slides/presentations/06A_Technical_Presentation_Student_Attrition.slides.html


In [8]:
from google.colab import files

files.download(
    "/content/capstone_slides/presentations/"
    "06A_Technical_Presentation_Student_Attrition.slides.html"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
css = """
<style>
/* Overall slide canvas */
.reveal .slides {
    text-align: left;
}

/* Slide positioning */
.reveal .slides section {
    text-align: left;
    top: 0 !important;
}

/* Typography */
.reveal {
    font-family: Arial, Helvetica, sans-serif;
    font-size: 30px;
    color: #1f2937;
}

.reveal h1 {
    font-size: 1.9em;
    font-weight: 700;
    color: #16324f;
    margin-bottom: 0.45em;
}

.reveal h2 {
    font-size: 1.25em;
    font-weight: 700;
    color: #24557a;
    margin-top: 0.5em;
    margin-bottom: 0.3em;
}

.reveal h3 {
    font-size: 1em;
    font-weight: 700;
    color: #374151;
}

.reveal p,
.reveal li {
    line-height: 1.35;
}

.reveal ul,
.reveal ol {
    margin-top: 0.25em;
}

.reveal li {
    margin-bottom: 0.18em;
}

/* Tables */
.reveal table {
    width: 100%;
    font-size: 0.72em;
    border-collapse: collapse;
}

.reveal table th {
    background: #16324f;
    color: white;
    padding: 10px 12px;
}

.reveal table td {
    padding: 9px 12px;
    border-bottom: 1px solid #d1d5db;
}

/* Quotes / takeaway boxes */
.reveal blockquote {
    width: auto;
    margin: 0.7em 0;
    padding: 0.55em 0.8em;
    background: #eef4f8;
    border-left: 5px solid #24557a;
    box-shadow: none;
    font-style: normal;
}

/* Inline code */
.reveal code {
    color: #16324f;
    background: #f3f4f6;
    padding: 0.05em 0.2em;
}

/* Strong text */
.reveal strong {
    color: #16324f;
}
</style>
"""

html_path = (
    "/content/capstone_slides/presentations/"
    "06A_Technical_Presentation_Student_Attrition.slides.html"
)

with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace("</head>", css + "\n</head>")

with open(html_path, "w", encoding="utf-8") as f:
    f.write(html)

print("Presentation styling applied.")

Presentation styling applied.


In [10]:
from google.colab import files

files.download(
    "/content/capstone_slides/presentations/"
    "06A_Technical_Presentation_Student_Attrition.slides.html"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
html_path = (
    "/content/capstone_slides/presentations/"
    "06A_Technical_Presentation_Student_Attrition.slides.html"
)

fit_css = """
<style>

/* More compact slide typography */
.reveal {
    font-size: 22px !important;
}

.reveal h1 {
    font-size: 1.65em !important;
    margin-top: 0 !important;
    margin-bottom: 0.35em !important;
}

.reveal h2 {
    font-size: 1.05em !important;
    margin-top: 0.35em !important;
    margin-bottom: 0.2em !important;
}

.reveal h3 {
    font-size: 0.9em !important;
    margin-top: 0.25em !important;
    margin-bottom: 0.15em !important;
}

.reveal p {
    margin: 0.3em 0 !important;
}

.reveal ul,
.reveal ol {
    margin-top: 0.15em !important;
    margin-bottom: 0.3em !important;
}

.reveal li {
    margin-bottom: 0.10em !important;
    line-height: 1.25 !important;
}

.reveal blockquote {
    margin: 0.4em 0 !important;
    padding: 0.4em 0.65em !important;
}

.reveal table {
    font-size: 0.68em !important;
    margin-top: 0.3em !important;
    margin-bottom: 0.3em !important;
}

.reveal table th,
.reveal table td {
    padding: 6px 8px !important;
}

</style>
"""

with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace("</head>", fit_css + "\n</head>")

with open(html_path, "w", encoding="utf-8") as f:
    f.write(html)

print("Slide-fit styling applied.")

Slide-fit styling applied.


In [12]:
from google.colab import files

files.download(
    "/content/capstone_slides/presentations/"
    "06A_Technical_Presentation_Student_Attrition.slides.html"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from google.colab import files

uploaded_latest = files.upload()

Saving 06A_Technical_Presentation_Student_Attrition.ipynb to 06A_Technical_Presentation_Student_Attrition (3).ipynb


In [23]:
import json
import subprocess
from pathlib import Path
from google.colab import files

uploaded_name = next(iter(uploaded_latest.keys()))
source_path = Path("/content") / uploaded_name
preview_ipynb = Path("/content/technical_preview.ipynb")
preview_html = Path("/content/technical_preview.slides.html")

# Load latest notebook
with open(source_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Mark first 10 Markdown cells as slides
slide_count = 0

for cell in notebook["cells"]:
    cell.setdefault("metadata", {})
    cell["metadata"].setdefault("slideshow", {})

    if cell["cell_type"] == "markdown" and slide_count < 10:
        cell["metadata"]["slideshow"]["slide_type"] = "slide"
        slide_count += 1
    else:
        cell["metadata"]["slideshow"]["slide_type"] = "skip"

with open(preview_ipynb, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=1, ensure_ascii=False)

# Convert to Reveal.js
subprocess.run([
    "jupyter", "nbconvert",
    "--to", "slides",
    str(preview_ipynb),
    "--output", "technical_preview",
    "--output-dir", "/content"
], check=True)

# Apply our presentation styling
with open(preview_html, "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace("</head>", css + fit_css + "\n</head>")

with open(preview_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Latest preview created with {slide_count} slides.")
files.download(str(preview_html))

Latest preview created with 10 slides.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
!ls -la /content/technical_preview*

-rw-r--r-- 1 root root  34658 Oct  6 13:35 /content/technical_preview.ipynb
-rw-r--r-- 1 root root 287223 Oct  6 13:35 /content/technical_preview.slides.html.slides.html


In [16]:
from pathlib import Path
from google.colab import files

actual_html = Path("/content/technical_preview.slides.html.slides.html")
fixed_html = Path("/content/technical_preview.slides.html")

actual_html.rename(fixed_html)

with open(fixed_html, "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace("</head>", css + "\n</head>")

with open(fixed_html, "w", encoding="utf-8") as f:
    f.write(html)

print("Styled preview ready.")
files.download(str(fixed_html))

Styled preview ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>